# 🤖 DeepSeek Crypto Master - Complete Analysis Platform
## Leveraging All Modules + TradingView + Interactive Chatbot

**This notebook combines:**
- 📊 **Crypto Analysis** (`crypto_analysis.py`)
- 🔍 **Sentiment Analysis** (`sentiment_analysis.py`) 
- 🌐 **Web Scraping** (`web_scraping.py`)
- 📈 **TradingView Technical Analysis**
- 🤖 **Interactive DeepSeek Chatbot**

**Powered by DeepSeek AI for comprehensive crypto market analysis**

## 1. Setup and Imports

In [1]:
# Core imports
import sys
import os
import requests
import json
import time
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Data processing
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('dark_background')

# Web scraping
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Environment
from dotenv import load_dotenv
load_dotenv()

print("✅ Core libraries imported successfully!")
print(f"📅 Setup Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Core libraries imported successfully!
📅 Setup Date: 2025-09-03 20:15:11


## 2. Import Custom Modules

In [2]:
# Import our custom modules
try:
    from crypto_analysis import CryptoAnalyzer
    print("✅ CryptoAnalyzer imported")
    CRYPTO_ANALYSIS_AVAILABLE = True
except ImportError as e:
    print(f"❌ CryptoAnalyzer import failed: {e}")
    CRYPTO_ANALYSIS_AVAILABLE = False

try:
    from sentiment_analysis import CryptoSentimentAnalyzer
    print("✅ CryptoSentimentAnalyzer imported")
    SENTIMENT_ANALYSIS_AVAILABLE = True
except ImportError as e:
    print(f"❌ CryptoSentimentAnalyzer import failed: {e}")
    SENTIMENT_ANALYSIS_AVAILABLE = False

try:
    from web_scraping import WebScraper
    print("✅ WebScraper imported")
    WEB_SCRAPING_AVAILABLE = True
except ImportError as e:
    print(f"❌ WebScraper import failed: {e}")
    WEB_SCRAPING_AVAILABLE = False

# Initialize modules
if CRYPTO_ANALYSIS_AVAILABLE:
    crypto_analyzer = CryptoAnalyzer()
    print("🔧 CryptoAnalyzer initialized")

if SENTIMENT_ANALYSIS_AVAILABLE:
    sentiment_analyzer = CryptoSentimentAnalyzer()
    print("🔧 SentimentAnalyzer initialized")

if WEB_SCRAPING_AVAILABLE:
    web_scraper = WebScraper()
    print("🔧 WebScraper initialized")

print(f"\n🎉 Module Status:")
print(f"📊 Crypto Analysis: {'✅' if CRYPTO_ANALYSIS_AVAILABLE else '❌'}")
print(f"🔍 Sentiment Analysis: {'✅' if SENTIMENT_ANALYSIS_AVAILABLE else '❌'}")
print(f"🌐 Web Scraping: {'✅' if WEB_SCRAPING_AVAILABLE else '❌'}")

✅ CryptoAnalyzer imported
✅ CryptoSentimentAnalyzer imported
✅ WebScraper imported
🔧 CryptoAnalyzer initialized
🔍 Crypto Sentiment Analyzer Initialized
🔑 Tavily: ✅
🔑 OpenRouter: ✅
📊 VADER: ✅
📊 TextBlob: ✅
🔧 SentimentAnalyzer initialized
🔧 WebScraper initialized

🎉 Module Status:
📊 Crypto Analysis: ✅
🔍 Sentiment Analysis: ✅
🌐 Web Scraping: ✅


## 3. TradingView Technical Analysis Scraper

In [3]:
class TradingViewScraper:
    """Scrape technical analysis from TradingView"""
    
    def __init__(self):
        self.base_url = "https://www.tradingview.com/ideas/technicalanalysis/"
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        })
    
    def setup_selenium_driver(self):
        """Setup Selenium WebDriver for dynamic content"""
        try:
            chrome_options = Options()
            chrome_options.add_argument('--headless')
            chrome_options.add_argument('--no-sandbox')
            chrome_options.add_argument('--disable-dev-shm-usage')
            chrome_options.add_argument('--disable-gpu')
            chrome_options.add_argument('--window-size=1920,1080')
            
            driver = webdriver.Chrome(options=chrome_options)
            return driver
        except Exception as e:
            print(f"❌ Selenium setup failed: {e}")
            return None
    
    def scrape_crypto_ideas(self, crypto_symbol="BTC", max_ideas=5):
        """Scrape crypto technical analysis ideas from TradingView"""
        print(f"📈 Scraping TradingView ideas for {crypto_symbol}...")
        
        try:
            # Try with requests first (faster)
            url = f"{self.base_url}?sort=recent&symbol={crypto_symbol}USD"
            response = self.session.get(url, timeout=15)
            
            if response.status_code == 200:
                soup = BeautifulSoup(response.content, 'html.parser')
                ideas = self._extract_ideas_from_soup(soup, max_ideas)
                
                if ideas:
                    print(f"✅ Found {len(ideas)} ideas using requests")
                    return ideas
            
            # Fallback to Selenium for dynamic content
            print("🔄 Trying with Selenium...")
            return self._scrape_with_selenium(crypto_symbol, max_ideas)
            
        except Exception as e:
            print(f"❌ Error scraping TradingView: {e}")
            return []
    
    def _extract_ideas_from_soup(self, soup, max_ideas):
        """Extract ideas from BeautifulSoup object"""
        ideas = []
        
        # Look for idea containers (TradingView structure may vary)
        idea_containers = soup.find_all(['div', 'article'], class_=lambda x: x and ('idea' in x.lower() or 'post' in x.lower()))[:max_ideas]
        
        for container in idea_containers:
            try:
                # Extract title
                title_elem = container.find(['h1', 'h2', 'h3', 'a'], class_=lambda x: x and 'title' in x.lower())
                title = title_elem.get_text().strip() if title_elem else "No title"
                
                # Extract author
                author_elem = container.find(['span', 'div'], class_=lambda x: x and 'author' in x.lower())
                author = author_elem.get_text().strip() if author_elem else "Unknown"
                
                # Extract content/description
                content_elem = container.find(['p', 'div'], class_=lambda x: x and ('content' in x.lower() or 'description' in x.lower()))
                content = content_elem.get_text().strip() if content_elem else ""
                
                # Extract link
                link_elem = container.find('a', href=True)
                link = link_elem['href'] if link_elem else ""
                if link and not link.startswith('http'):
                    link = f"https://www.tradingview.com{link}"
                
                if title and title != "No title":
                    ideas.append({
                        'title': title,
                        'author': author,
                        'content': content[:200] + "..." if len(content) > 200 else content,
                        'link': link,
                        'source': 'TradingView',
                        'timestamp': datetime.now().isoformat()
                    })
                    
            except Exception as e:
                continue
        
        return ideas
    
    def _scrape_with_selenium(self, crypto_symbol, max_ideas):
        """Scrape using Selenium for dynamic content"""
        driver = self.setup_selenium_driver()
        if not driver:
            return []
        
        try:
            url = f"{self.base_url}?sort=recent&symbol={crypto_symbol}USD"
            driver.get(url)
            
            # Wait for content to load
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "body"))
            )
            
            time.sleep(3)  # Additional wait for dynamic content
            
            # Get page source and parse
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            ideas = self._extract_ideas_from_soup(soup, max_ideas)
            
            print(f"✅ Found {len(ideas)} ideas using Selenium")
            return ideas
            
        except Exception as e:
            print(f"❌ Selenium scraping failed: {e}")
            return []
        finally:
            driver.quit()
    
    def get_mock_technical_analysis(self, crypto_symbol):
        """Provide mock technical analysis if scraping fails"""
        return [
            {
                'title': f'{crypto_symbol} Technical Analysis - Bullish Momentum',
                'author': 'TradingView Analyst',
                'content': f'{crypto_symbol} showing strong technical indicators with RSI in neutral zone and MACD showing positive divergence. Support levels holding well.',
                'link': f'https://www.tradingview.com/ideas/technicalanalysis/?symbol={crypto_symbol}USD',
                'source': 'TradingView (Mock)',
                'timestamp': datetime.now().isoformat()
            },
            {
                'title': f'{crypto_symbol} Price Action Analysis',
                'author': 'Technical Analyst',
                'content': f'Current {crypto_symbol} price action suggests consolidation phase with potential breakout. Key resistance and support levels identified.',
                'link': f'https://www.tradingview.com/ideas/technicalanalysis/?symbol={crypto_symbol}USD',
                'source': 'TradingView (Mock)',
                'timestamp': datetime.now().isoformat()
            }
        ]

# Initialize TradingView scraper
tradingview_scraper = TradingViewScraper()
print("✅ TradingView scraper initialized")

✅ TradingView scraper initialized


## 4. DeepSeek AI Integration

In [4]:
class DeepSeekAI:
    """DeepSeek AI integration for comprehensive analysis"""
    
    def __init__(self):
        self.api_key = os.getenv('OPENROUTER_API_KEY')
        self.base_url = "https://openrouter.ai/api/v1/chat/completions"
        self.model = "deepseek/deepseek-chat"
        
        if not self.api_key:
            print("⚠️  OpenRouter API key not found. Add OPENROUTER_API_KEY to .env file")
            self.available = False
        else:
            print("✅ DeepSeek AI initialized")
            self.available = True
    
    def analyze_comprehensive_data(self, crypto_name, price_data, sentiment_data, technical_analysis, web_data):
        """Comprehensive analysis using all available data"""
        if not self.available:
            return {'success': False, 'error': 'DeepSeek AI not available'}
        
        # Prepare comprehensive prompt
        prompt = f"""
As a professional cryptocurrency analyst, provide a comprehensive analysis of {crypto_name} based on the following data:

PRICE DATA:
{json.dumps(price_data, indent=2) if price_data else 'No price data available'}

SENTIMENT ANALYSIS:
{json.dumps(sentiment_data, indent=2) if sentiment_data else 'No sentiment data available'}

TECHNICAL ANALYSIS (TradingView):
{json.dumps(technical_analysis, indent=2) if technical_analysis else 'No technical analysis available'}

WEB DATA:
{json.dumps(web_data, indent=2) if web_data else 'No web data available'}

Please provide:
1. Overall market assessment
2. Key technical levels and indicators
3. Sentiment analysis interpretation
4. Trading recommendation (BUY/HOLD/SELL)
5. Risk assessment
6. Price targets and stop losses
7. Confidence level (1-10)

Format your response in a clear, professional manner suitable for trading decisions.
"""
        
        return self._make_api_call(prompt)
    
    def chat_response(self, user_message, context_data=None):
        """Interactive chat response with context"""
        if not self.available:
            return {'success': False, 'error': 'DeepSeek AI not available'}
        
        # Add context if available
        context_prompt = ""
        if context_data:
            # Convert datetime objects to strings for JSON serialization
            def convert_datetime(obj):
                if isinstance(obj, dict):
                    return {k: convert_datetime(v) for k, v in obj.items()}
                elif isinstance(obj, list):
                    return [convert_datetime(item) for item in obj]
                elif hasattr(obj, 'isoformat'):  # datetime objects
                    return obj.isoformat()
                else:
                    return obj
            
            try:
                serializable_context = convert_datetime(context_data)
                context_prompt = f"""
CONTEXT DATA:
{json.dumps(serializable_context, indent=2, default=str)}

"""
            except Exception as e:
                context_prompt = f"""
CONTEXT DATA: (Serialization error: {str(e)})
Basic context available but not serializable.

"""
        
        prompt = f"""
You are a professional cryptocurrency analyst and trading expert. Answer the user's question based on your expertise and any provided context data.

{context_prompt}
USER QUESTION: {user_message}

Provide a helpful, accurate, and professional response. If you need more specific data to give a complete answer, mention what additional information would be helpful.
"""
        
        return self._make_api_call(prompt)
    
    def _make_api_call(self, prompt):
        """Make API call to DeepSeek"""
        try:
            headers = {
                "Authorization": f"Bearer {self.api_key}",
                "Content-Type": "application/json"
            }
            
            data = {
                "model": self.model,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 2000,
                "temperature": 0.7
            }
            
            response = requests.post(self.base_url, headers=headers, json=data, timeout=60)
            
            if response.status_code == 200:
                result = response.json()
                content = result['choices'][0]['message']['content']
                return {
                    'success': True,
                    'content': content,
                    'timestamp': datetime.now().isoformat()
                }
            else:
                return {
                    'success': False,
                    'error': f"API call failed: {response.status_code} - {response.text}"
                }
                
        except Exception as e:
            return {
                'success': False,
                'error': f"API call error: {str(e)}"
            }

# Initialize DeepSeek AI
deepseek_ai = DeepSeekAI()
print(f"🤖 DeepSeek AI Status: {'✅ Ready' if deepseek_ai.available else '❌ Not Available'}")

✅ DeepSeek AI initialized
🤖 DeepSeek AI Status: ✅ Ready


## 5. Master Analysis Function

In [5]:
def comprehensive_crypto_analysis(crypto_name, max_articles=5, max_tradingview_ideas=3):
    """
    Comprehensive crypto analysis using all available modules
    """
    print(f"🚀 Starting comprehensive analysis for {crypto_name}")
    print("=" * 80)
    
    analysis_results = {
        'crypto_name': crypto_name,
        'timestamp': datetime.now().isoformat(),
        'price_data': None,
        'sentiment_data': None,
        'technical_analysis': None,
        'web_data': None,
        'deepseek_analysis': None
    }
    
    # 1. Crypto Analysis (Price + Technical)
    if CRYPTO_ANALYSIS_AVAILABLE:
        print("\n1️⃣ Getting price and technical analysis...")
        try:
            price_result = crypto_analyzer.process_query(f"What's the price of {crypto_name}?")
            if price_result.get('success'):
                analysis_results['price_data'] = price_result
                print("   ✅ Price analysis completed")
            else:
                print(f"   ❌ Price analysis failed: {price_result.get('error')}")
        except Exception as e:
            print(f"   ❌ Price analysis error: {e}")
    else:
        print("\n1️⃣ ❌ Crypto analysis module not available")
    
    # 2. Sentiment Analysis
    if SENTIMENT_ANALYSIS_AVAILABLE:
        print("\n2️⃣ Performing sentiment analysis...")
        try:
            sentiment_result = sentiment_analyzer.analyze_cryptocurrency(crypto_name, max_articles)
            if sentiment_result.get('success'):
                analysis_results['sentiment_data'] = sentiment_result
                print("   ✅ Sentiment analysis completed")
            else:
                print(f"   ❌ Sentiment analysis failed: {sentiment_result.get('error')}")
        except Exception as e:
            print(f"   ❌ Sentiment analysis error: {e}")
    else:
        print("\n2️⃣ ❌ Sentiment analysis module not available")
    
    # 3. TradingView Technical Analysis
    print("\n3️⃣ Scraping TradingView technical analysis...")
    try:
        # Extract ticker symbol from crypto name
        ticker_map = {
            'bitcoin': 'BTC', 'ethereum': 'ETH', 'solana': 'SOL',
            'cardano': 'ADA', 'polygon': 'MATIC', 'chainlink': 'LINK'
        }
        ticker = ticker_map.get(crypto_name.lower(), crypto_name.upper()[:3])
        
        technical_ideas = tradingview_scraper.scrape_crypto_ideas(ticker, max_tradingview_ideas)
        
        if not technical_ideas:
            print("   ⚠️  Using mock technical analysis data")
            technical_ideas = tradingview_scraper.get_mock_technical_analysis(ticker)
        
        analysis_results['technical_analysis'] = technical_ideas
        print(f"   ✅ Found {len(technical_ideas)} technical analysis ideas")
        
    except Exception as e:
        print(f"   ❌ TradingView scraping error: {e}")
    
    # 4. Web Scraping
    if WEB_SCRAPING_AVAILABLE:
        print("\n4️⃣ Performing additional web scraping...")
        try:
            # Get crypto news
            news_data = web_scraper.scrape_crypto_news(limit=3)
            
            # Get price data from multiple sources
            price_data = web_scraper.scrape_crypto_prices([ticker])
            
            analysis_results['web_data'] = {
                'news': news_data,
                'prices': price_data
            }
            print(f"   ✅ Scraped {len(news_data)} news items and {len(price_data)} price entries")
            
        except Exception as e:
            print(f"   ❌ Web scraping error: {e}")
    else:
        print("\n4️⃣ ❌ Web scraping module not available")
    
    # 5. DeepSeek Comprehensive Analysis
    if deepseek_ai.available:
        print("\n5️⃣ Generating DeepSeek AI comprehensive analysis...")
        try:
            deepseek_result = deepseek_ai.analyze_comprehensive_data(
                crypto_name,
                analysis_results['price_data'],
                analysis_results['sentiment_data'],
                analysis_results['technical_analysis'],
                analysis_results['web_data']
            )
            
            if deepseek_result.get('success'):
                analysis_results['deepseek_analysis'] = deepseek_result
                print("   ✅ DeepSeek analysis completed")
            else:
                print(f"   ❌ DeepSeek analysis failed: {deepseek_result.get('error')}")
                
        except Exception as e:
            print(f"   ❌ DeepSeek analysis error: {e}")
    else:
        print("\n5️⃣ ❌ DeepSeek AI not available")
    
    print("\n" + "=" * 80)
    print("🎉 Comprehensive analysis completed!")
    
    return analysis_results

def display_analysis_results(results):
    """
    Display comprehensive analysis results in a formatted way
    """
    print(f"\n📊 COMPREHENSIVE ANALYSIS RESULTS FOR {results['crypto_name'].upper()}")
    print("=" * 80)
    print(f"📅 Analysis Time: {results['timestamp'][:19]}")
    
    # Price Data
    if results['price_data']:
        price_data = results['price_data'].get('price_data', {})
        if price_data:
            print(f"\n💰 PRICE DATA:")
            print(f"   Current Price: ${price_data.get('current_price', 'N/A'):,.2f}" if isinstance(price_data.get('current_price'), (int, float)) else f"   Current Price: {price_data.get('current_price', 'N/A')}")
            print(f"   24h Change: {price_data.get('price_change_24h', 'N/A'):+.2f}%" if isinstance(price_data.get('price_change_24h'), (int, float)) else f"   24h Change: {price_data.get('price_change_24h', 'N/A')}")
    
    # Sentiment Data
    if results['sentiment_data']:
        sentiment = results['sentiment_data']
        print(f"\n🔍 SENTIMENT ANALYSIS:")
        print(f"   Articles Analyzed: {sentiment.get('articles_analyzed', 'N/A')}")
        
        if 'vader' in sentiment:
            vader = sentiment['vader']
            print(f"   VADER Sentiment: {vader['sentiment']} ({vader['average_score']:.3f})")
        
        if 'textblob' in sentiment:
            textblob = sentiment['textblob']
            print(f"   TextBlob Sentiment: {textblob['sentiment']} ({textblob['average_score']:.3f})")
    
    # Technical Analysis
    if results['technical_analysis']:
        print(f"\n📈 TRADINGVIEW TECHNICAL ANALYSIS:")
        for i, idea in enumerate(results['technical_analysis'][:2], 1):
            print(f"   {i}. {idea['title'][:60]}...")
            print(f"      Author: {idea['author']}")
    
    # DeepSeek Analysis
    if results['deepseek_analysis'] and results['deepseek_analysis'].get('success'):
        print(f"\n🤖 DEEPSEEK AI ANALYSIS:")
        print("─" * 60)
        print(results['deepseek_analysis']['content'])
        print("─" * 60)
    
    print("\n" + "=" * 80)

print("✅ Master analysis functions defined")

✅ Master analysis functions defined


## 6. Run Comprehensive Analysis

In [6]:
# Configure analysis parameters
CRYPTO_TO_ANALYZE = "Bitcoin"  # Change this to analyze different cryptocurrencies
MAX_ARTICLES = 3               # Number of articles for sentiment analysis
MAX_TRADINGVIEW_IDEAS = 2      # Number of TradingView ideas to scrape

print(f"🎯 Analysis Configuration:")
print(f"   Cryptocurrency: {CRYPTO_TO_ANALYZE}")
print(f"   Max Articles: {MAX_ARTICLES}")
print(f"   Max TradingView Ideas: {MAX_TRADINGVIEW_IDEAS}")

# Run comprehensive analysis
analysis_results = comprehensive_crypto_analysis(
    CRYPTO_TO_ANALYZE, 
    MAX_ARTICLES, 
    MAX_TRADINGVIEW_IDEAS
)

# Display results
display_analysis_results(analysis_results)

# Store results for chatbot context
CURRENT_ANALYSIS_CONTEXT = analysis_results

🎯 Analysis Configuration:
   Cryptocurrency: Bitcoin
   Max Articles: 3
   Max TradingView Ideas: 2
🚀 Starting comprehensive analysis for Bitcoin

1️⃣ Getting price and technical analysis...
🔍 Processing query: 'What's the price of Bitcoin?'
1️⃣ Extracting cryptocurrency ticker...
✅ Extracted ticker: BTC
2️⃣ Fetching current price for BTC...
✅ Current price data fetched successfully!
   💰 Price: $112,168.27
   📈 24h Change: +1.01%
   📊 24h High/Low: $112,500.00 / $110,200.00
   📦 24h Volume: 12,977
3️⃣ Getting AI analysis for BTC...
✅ AI Analysis completed!
   ✅ Price analysis completed

2️⃣ Performing sentiment analysis...

🔍 Starting sentiment analysis for: Bitcoin
🔍 Searching Tavily: 'Bitcoin cryptocurrency news sentiment price analysis'
✅ Found 1 articles from Tavily
📰 Scraping CoinDesk...
   ⚠️  No relevant articles found
📰 Scraping CoinTelegraph...
   ✅ Found 1 relevant articles
📰 Scraping BeInCrypto...
   ⚠️  No relevant articles found
📰 Scraping U.Today...
   ⚠️  No relevant ar

## 7. Interactive DeepSeek Chatbot

In [ ]:
# Interactive Chatbot Cell
print("🤖 DEEPSEEK CRYPTO CHATBOT")
print("=" * 50)
print("Ask me anything about crypto analysis!")
print("Examples:")
print("- 'What's your recommendation for Bitcoin?'")
print("- 'Explain the sentiment analysis results'")
print("- 'What are the key technical levels?'")
print("- 'Should I buy or sell now?'")
print("\nType 'quit' to exit the chat")
print("-" * 50)

# Chat loop
chat_history = []

while True:
    try:
        # Get user input
        user_message = input("\n💬 You: ").strip()
        
        if user_message.lower() in ['quit', 'exit', 'bye']:
            print("👋 Thanks for using DeepSeek Crypto Chatbot!")
            break
        
        if not user_message:
            print("⚠️  Please enter a message")
            continue
        
        # Add to chat history
        chat_history.append({"role": "user", "message": user_message, "timestamp": datetime.now().isoformat()})
        
        print("\n🤖 DeepSeek: Thinking...")
        
        # Get AI response with context
        if deepseek_ai.available:
            # Prepare context data
            context_data = {
                'current_analysis': CURRENT_ANALYSIS_CONTEXT,
                'chat_history': chat_history[-3:] if len(chat_history) > 3 else chat_history  # Last 3 messages for context
            }
            
            response = deepseek_ai.chat_response(user_message, context_data)
            
            if response.get('success'):
                ai_message = response['content']
                print(f"\n🤖 DeepSeek: {ai_message}")
                
                # Add AI response to history
                chat_history.append({"role": "assistant", "message": ai_message, "timestamp": datetime.now().isoformat()})
            else:
                print(f"\n❌ Error: {response.get('error')}")
        else:
            print("\n❌ DeepSeek AI is not available. Please check your OPENROUTER_API_KEY in .env file")
    
    except KeyboardInterrupt:
        print("\n\n👋 Chat interrupted. Thanks for using DeepSeek Crypto Chatbot!")
        break
    except Exception as e:
        print(f"\n❌ Chat error: {e}")
        continue

print("\n📊 Chat session ended.")
print(f"💬 Total messages exchanged: {len(chat_history)}")

🤖 DEEPSEEK CRYPTO CHATBOT
Ask me anything about crypto analysis!
Examples:
- 'What's your recommendation for Bitcoin?'
- 'Explain the sentiment analysis results'
- 'What are the key technical levels?'
- 'Should I buy or sell now?'

Type 'quit' to exit the chat
--------------------------------------------------

🤖 DeepSeek: Thinking...

🤖 DeepSeek: To provide a comprehensive analysis of Solana (SOL), I would need specific data points such as its current price, 24-hour price change, trading volume, market sentiment, technical indicators, and recent news. Unfortunately, the context data provided focuses on Bitcoin (BTC) and does not include information about Solana.

Here’s a general framework for analyzing Solana based on typical factors:

### 1. **Price Data**
   - **Current Price**: The current trading price of Solana.
   - **Price Change (24h)**: The percentage change in price over the last 24 hours.
   - **High/Low (24h)**: The highest and lowest price points in the last 24 hours.
  